# Diagnóstico exploratorio del FAIL held-out lineal

Este notebook estudia el único fallo del protocolo pareado: pureza random `47.92%` en seed 7 frente al corte absoluto `50%`. Reutiliza deliberadamente el test ya consumido y, por lo tanto, **no puede cambiar el FAIL ni producir una nueva decisión confirmatoria**.

## Preguntas descriptivas

1. ¿Qué regímenes explican los errores después de alinear los IDs arbitrarios de K-means?
2. ¿El resultado depende mucho de `random_state` de K-means manteniendo `n_init = 20`?
3. ¿Qué ocurre con `sine_med_freq` y `sine_low_amp`, que son observacionalmente equivalentes bajo normalización por secuencia?

Las respuestas sirven para diseñar datos nuevos. No se agregan umbrales ni se reevalúa el gate consumido.

In [ ]:
# ruff: noqa: E402, E501
import json
import platform
import random
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "No se encontró la raíz del repositorio."
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

from koopman_jepa.analysis import clustering_diagnostics
from koopman_jepa.paper_config import load_paper_linear_random_heldout_config
from koopman_jepa.paper_data import PAPER_REGIME_NAMES, PaperRegimeDataset, generate_paper_master
from koopman_jepa.paper_model import PaperTemporalJEPA
from koopman_jepa.paper_training import (
    make_paper_loader,
    run_paper_train_validation_with_checkpoint,
    verify_paper_checkpoint_replay,
)

plt.style.use("seaborn-v0_8-whitegrid")
print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
config_path = ROOT / "configs" / "paper_linear_random_heldout_smoke.yaml"
config = load_paper_linear_random_heldout_config(config_path)
identity_model_config = replace(config.model, linear_initialization="identity")
torch.use_deterministic_algorithms(True)
kmeans_random_states = tuple(range(20))

print(config_path.relative_to(ROOT))
print(json.dumps(asdict(config), indent=2))
print(f"K-means random states descriptivos: {kmeans_random_states}")

## Reconstrucción del resultado consumido

Se reconstruyen los diez checkpoints con train/validation. El acceso exploratorio al test sólo se habilita si todos los epochs y métricas recargadas coinciden y las inicializaciones de encoder siguen pareadas.

In [ ]:
train_dataset = PaperRegimeDataset(config.data, "train", generate_paper_master)
validation_dataset = PaperRegimeDataset(config.data, "val", generate_paper_master)
train_keys = {train_dataset.sample_key(index) for index in range(len(train_dataset))}
validation_keys = {validation_dataset.sample_key(index) for index in range(len(validation_dataset))}
assert train_keys.isdisjoint(validation_keys)
print(f"Train/validation: {len(train_dataset)}/{len(validation_dataset)}")
print("Test consumido todavía no fue reconstruido en esta ejecución.")

In [ ]:
def clone_initial_state(model):
    return {
        "online": {name: tensor.detach().cpu().clone() for name, tensor in model.online_encoder.state_dict().items()},
        "target": {name: tensor.detach().cpu().clone() for name, tensor in model.target_encoder.state_dict().items()},
        "predictor": model.predictor.matrix.detach().cpu().clone(),
    }


def encoders_match(observed, reference):
    return all(
        torch.equal(tensor, reference[module_name][name])
        for module_name in ("online", "target")
        for name, tensor in observed[module_name].items()
    )


def replay_condition(name, model_config, replay_config, reference_states=None):
    expected_epochs = dict(zip(config.sweep.seeds, replay_config.expected_epochs, strict=True))
    models, results, initial_states, pairing_checks, times = {}, {}, {}, {}, {}
    for seed in config.sweep.seeds:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        run_config = replace(config.train, seed=seed)
        model = PaperTemporalJEPA(model_config)
        initial_state = clone_initial_state(model)
        initial_states[seed] = initial_state
        pairing_checks[seed] = reference_states is None or (
            encoders_match(initial_state, reference_states[seed])
            and not torch.equal(initial_state["predictor"], reference_states[seed]["predictor"])
        )
        started = time.perf_counter()
        checkpoint_run = run_paper_train_validation_with_checkpoint(
            model, train_dataset, validation_dataset, run_config, config.checkpoint_gate
        )
        replay = verify_paper_checkpoint_replay(
            model, checkpoint_run, validation_dataset, run_config, replay_config,
            expected_epoch=expected_epochs[seed],
        )
        models[seed], results[seed] = model, replay
        times[seed] = time.perf_counter() - started
        selected = checkpoint_run.selection.epoch if checkpoint_run.selection else None
        print(f"{name} seed={seed} selected={selected} pairing={pairing_checks[seed]} replay={'PASS' if replay.passed else 'FAIL'} time={times[seed]:.2f}s")
    return models, results, initial_states, pairing_checks, times


identity_models, identity_replays, identity_initial_states, _, identity_times = replay_condition(
    "identity", identity_model_config, config.identity_replay
)
random_models, random_replays, _, paired_initializations, random_times = replay_condition(
    "random", config.model, config.random_replay, identity_initial_states
)
diagnostic_replay_open = (
    set(identity_replays) == set(config.sweep.seeds)
    and set(random_replays) == set(config.sweep.seeds)
    and all(result.passed for result in identity_replays.values())
    and all(result.passed for result in random_replays.values())
    and all(paired_initializations.values())
)
assert diagnostic_replay_open, "No se reconstruyó exactamente el resultado consumido."
print(f"Replay diagnóstico: PASS; tiempo total={sum(identity_times.values()) + sum(random_times.values()):.2f}s")

In [ ]:
assert globals().get("diagnostic_replay_open", False)
test_dataset = PaperRegimeDataset(config.data, "test", generate_paper_master)
test_keys = {test_dataset.sample_key(index) for index in range(len(test_dataset))}
assert len(test_dataset) == 144
assert train_keys.isdisjoint(test_keys) and validation_keys.isdisjoint(test_keys)
print("Test ya consumido reconstruido para diagnóstico descriptivo: 144 secuencias.")

In [ ]:
@torch.no_grad()
def collect_online_embeddings(model, dataset, run_config):
    device = torch.device(run_config.device)
    model.to(device).eval()
    loader = make_paper_loader(dataset, run_config, shuffle=False)
    embedding_chunks, label_chunks = [], []
    for context, _, labels in loader:
        embedding_chunks.append(model.online_encoder(context.to(device)).cpu().numpy())
        label_chunks.append(labels.numpy())
    return np.concatenate(embedding_chunks), np.concatenate(label_chunks)


embeddings_by_condition = {"identity": {}, "random": {}}
labels_reference = None
for condition, models in (("identity", identity_models), ("random", random_models)):
    for seed in config.sweep.seeds:
        embeddings, labels = collect_online_embeddings(
            models[seed], test_dataset, replace(config.train, seed=seed)
        )
        if labels_reference is None:
            labels_reference = labels
        else:
            assert np.array_equal(labels, labels_reference)
        embeddings_by_condition[condition][seed] = embeddings

expected_counts = {
    "identity": {5: 66, 6: 71, 7: 67, 8: 76, 9: 76},
    "random": {5: 73, 6: 72, 7: 69, 8: 74, 9: 74},
}
fixed_diagnostics = {"identity": {}, "random": {}}
for condition in fixed_diagnostics:
    for seed in config.sweep.seeds:
        result = clustering_diagnostics(
            embeddings_by_condition[condition][seed],
            labels_reference,
            len(PAPER_REGIME_NAMES),
            config.evaluation.kmeans_seed,
            n_init=config.evaluation.kmeans_n_init,
        )
        observed_count = int(round(result["kmeans_purity"] * len(labels_reference)))
        assert observed_count == expected_counts[condition][seed]
        fixed_diagnostics[condition][seed] = result
print("Conteos de pureza del resultado held-out reproducidos exactamente.")

In [ ]:
stability_purity = {"identity": {}, "random": {}}
stability_matched = {"identity": {}, "random": {}}
for condition in stability_purity:
    for seed in config.sweep.seeds:
        runs = [
            clustering_diagnostics(
                embeddings_by_condition[condition][seed],
                labels_reference,
                len(PAPER_REGIME_NAMES),
                kmeans_seed,
                n_init=config.evaluation.kmeans_n_init,
            )
            for kmeans_seed in kmeans_random_states
        ]
        stability_purity[condition][seed] = np.array([run["kmeans_purity"] for run in runs])
        stability_matched[condition][seed] = np.array([run["kmeans_matched_accuracy"] for run in runs])

normalized_confusions = {}
mean_recalls = {}
for condition in ("identity", "random"):
    confusions = np.stack([fixed_diagnostics[condition][seed]["aligned_confusion"] for seed in config.sweep.seeds])
    normalized = confusions / confusions.sum(axis=2, keepdims=True)
    normalized_confusions[condition] = normalized.mean(axis=0)
    mean_recalls[condition] = np.stack([fixed_diagnostics[condition][seed]["per_regime_recall"] for seed in config.sweep.seeds]).mean(axis=0)

pair_ids = np.array([1, 3])
pair_retention = {"identity": {}, "random": {}}
for condition in pair_retention:
    for seed in config.sweep.seeds:
        confusion = fixed_diagnostics[condition][seed]["aligned_confusion"]
        pair_retention[condition][seed] = (
            confusion[np.ix_(pair_ids, pair_ids)].sum() / confusion[pair_ids, :].sum()
        )
print("Diagnósticos de regímenes y estabilidad K-means calculados; no se aplica ningún gate.")

In [ ]:
print("seed fixed_I fixed_R delta  km_I_mean±sd  km_R_mean±sd  pair_I pair_R")
for seed in config.sweep.seeds:
    fixed_i = fixed_diagnostics["identity"][seed]["kmeans_purity"]
    fixed_r = fixed_diagnostics["random"][seed]["kmeans_purity"]
    stable_i = stability_purity["identity"][seed]
    stable_r = stability_purity["random"][seed]
    print(
        f"{seed:>4d} {fixed_i:>7.2%} {fixed_r:>7.2%} {fixed_r-fixed_i:>+6.2%} "
        f"{stable_i.mean():>6.2%}±{stable_i.std():.2%} "
        f"{stable_r.mean():>6.2%}±{stable_r.std():.2%} "
        f"{pair_retention['identity'][seed]:>6.2%} {pair_retention['random'][seed]:>6.2%}"
    )

recall_delta = mean_recalls["random"] - mean_recalls["identity"]
print("\nregime                         recall_I recall_R delta")
for regime_id in np.argsort(mean_recalls["random"]):
    print(
        f"{PAPER_REGIME_NAMES[regime_id]:<30} "
        f"{mean_recalls['identity'][regime_id]:>7.2%} "
        f"{mean_recalls['random'][regime_id]:>7.2%} "
        f"{recall_delta[regime_id]:>+7.2%}"
    )

In [ ]:
seeds = np.array(config.sweep.seeds)
fixed_identity = np.array([fixed_diagnostics["identity"][seed]["kmeans_purity"] for seed in seeds])
fixed_random = np.array([fixed_diagnostics["random"][seed]["kmeans_purity"] for seed in seeds])
stability_identity = np.stack([stability_purity["identity"][seed] for seed in seeds])
stability_random = np.stack([stability_purity["random"][seed] for seed in seeds])

fig, axes = plt.subplots(2, 3, figsize=(19, 11), constrained_layout=True)
width = 0.36
axes[0, 0].bar(seeds - width / 2, fixed_identity, width=width, label="identidad")
axes[0, 0].bar(seeds + width / 2, fixed_random, width=width, label="random")
axes[0, 0].axhline(0.50, color="tab:red", linestyle="--", label="corte consumido")
axes[0, 0].set(title="Pureza fija reproducida", xlabel="Seed del modelo", ylabel="Pureza")
axes[0, 0].legend()

axes[0, 1].errorbar(seeds - 0.08, stability_identity.mean(axis=1), yerr=stability_identity.std(axis=1), fmt="o", capsize=4, label="identidad")
axes[0, 1].errorbar(seeds + 0.08, stability_random.mean(axis=1), yerr=stability_random.std(axis=1), fmt="o", capsize=4, label="random")
axes[0, 1].axhline(0.50, color="tab:red", linestyle="--")
axes[0, 1].set(title="Sensibilidad a random_state de K-means", xlabel="Seed del modelo", ylabel="Media ± sd de pureza")
axes[0, 1].legend()

pair_i = np.array([pair_retention["identity"][seed] for seed in seeds])
pair_r = np.array([pair_retention["random"][seed] for seed in seeds])
axes[0, 2].bar(seeds - width / 2, pair_i, width=width, label="identidad")
axes[0, 2].bar(seeds + width / 2, pair_r, width=width, label="random")
axes[0, 2].set(title="Retención dentro del par indistinguible", xlabel="Seed", ylabel="Fracción predicha como 1 o 3")
axes[0, 2].legend()

for axis, condition in zip(axes[1, :2], ("identity", "random"), strict=True):
    image = axis.imshow(normalized_confusions[condition], vmin=0.0, vmax=1.0, cmap="Blues")
    axis.set(title=f"Confusión media alineada: {condition}", xlabel="Predicción", ylabel="Régimen real")
    axis.set_xticks(range(len(PAPER_REGIME_NAMES)), labels=range(len(PAPER_REGIME_NAMES)), fontsize=7)
    axis.set_yticks(range(len(PAPER_REGIME_NAMES)), labels=range(len(PAPER_REGIME_NAMES)), fontsize=7)
    fig.colorbar(image, ax=axis, fraction=0.046)

colors = np.where(recall_delta >= 0.0, "tab:green", "tab:red")
axes[1, 2].barh(np.arange(len(PAPER_REGIME_NAMES)), recall_delta, color=colors, alpha=0.8)
axes[1, 2].axvline(0.0, color="black", linewidth=1)
axes[1, 2].set_yticks(np.arange(len(PAPER_REGIME_NAMES)), labels=PAPER_REGIME_NAMES, fontsize=8)
axes[1, 2].set(title="Cambio de recall random − identidad", xlabel="Diferencia")
plt.show()

In [ ]:
seed7_i = fixed_diagnostics["identity"][7]
seed7_r = fixed_diagnostics["random"][7]
seed7_delta = seed7_r["per_regime_recall"] - seed7_i["per_regime_recall"]
worst_seed7_ids = np.argsort(seed7_delta)[:3]
worst_seed7_text = ", ".join(
    f"{PAPER_REGIME_NAMES[index]} ({seed7_delta[index]:+.1%})"
    for index in worst_seed7_ids
)
identity_stability_span = (stability_identity.min(), stability_identity.max())
random_stability_span = (stability_random.min(), stability_random.max())
pair_identity_mean = np.mean(list(pair_retention["identity"].values()))
pair_random_mean = np.mean(list(pair_retention["random"].values()))
lowest_random_ids = np.argsort(mean_recalls["random"])[:3]
lowest_random_text = ", ".join(
    f"{PAPER_REGIME_NAMES[index]} ({mean_recalls['random'][index]:.1%})"
    for index in lowest_random_ids
)

display(Markdown(f"""## Interpretación exploratoria

- Los conteos del protocolo consumido se reprodujeron exactamente: seed 7 random mantiene `69/144 = 47.92%`, frente a `67/144 = 46.53%` para identidad.
- Al variar sólo `random_state` de K-means con `n_init=20`, la pureza identidad abarca `{identity_stability_span[0]:.2%}–{identity_stability_span[1]:.2%}` y random `{random_stability_span[0]:.2%}–{random_stability_span[1]:.2%}`. Esto cuantifica sensibilidad algorítmica, pero no reemplaza el valor congelado.
- Los tres recalls random medios más bajos son: {lowest_random_text}.
- En seed 7, las mayores caídas random−identidad ocurren en: {worst_seed7_text}.
- Para los dos regímenes observacionalmente equivalentes, la fracción media que permanece dentro del par es `{pair_identity_mean:.1%}` en identidad y `{pair_random_mean:.1%}` en random. Esta métrica acepta confundir uno con el otro y separa esa ambigüedad inevitable de errores hacia familias distintas.

### Lectura correcta

Este análisis localiza fuentes de error y estima cuánto varía K-means sobre los mismos embeddings. **No convierte el resultado en PASS**, no modifica el corte `50%` y no genera una nueva evaluación confirmatoria. Cualquier criterio aprendido aquí debe probarse únicamente en una realización de datos nueva y previamente congelada.
"""))